# 03 - Full Training (Qwen3.5-4B-Base + QLoRA, 269K Vietnamese)

Full QLoRA training: 269K train samples, 3 epochs, batch 32 x accum 2 (eff=64).
MaeEvalCallback selects best checkpoint by generative MAE on 500 val items every 500 steps.
After training: reload best_mae_checkpoint and push to HF Hub.

ETA: ~7-10h on RTX 5090 32GB.

In [ ]:
# Cell 1 - Imports + constants
import os, sys, torch, wandb
sys.path.insert(0, os.path.abspath("."))

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer

from utils.items_vn import load_items, DATASET_NAME
from utils.training_utils import (
    get_bnb_config, get_lora_config, get_sft_config,
    DataCollatorCompletionOnly, MaeEvalCallback,
    MAX_SEQ_LENGTH, VAL_EVAL_SIZE, RESPONSE_TEMPLATE,
)

BASE_MODEL    = "Qwen/Qwen3.5-4B-Base"
OUTPUT_DIR    = "outputs/qwen_v2"
HUB_MODEL_ID  = "SeanSunny/qwen3.5-4b-vn-pricer-v2"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU:  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2 - Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Eos token:  '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")

template_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
print(f"RESPONSE_TEMPLATE ids: {template_ids}")
print(f"RESPONSE_TEMPLATE decoded: '{tokenizer.decode(template_ids)}'")

In [ ]:
# Cell 3 - Load + format + tokenize dataset
def format_and_tokenize(example):
    text = example["prompt"] + example["completion"] + tokenizer.eos_token
    return tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

print("Loading train split...")
raw_train = load_dataset(DATASET_NAME, split="train")
print(f"  Raw train: {len(raw_train):,} rows")

train_ds = raw_train.map(
    format_and_tokenize,
    batched=True,
    remove_columns=raw_train.column_names,
    desc="Tokenising train",
)
print(f"  Tokenised train: {len(train_ds):,} rows")

sample = train_ds[0]
print(f"  input_ids length sample: {len(sample['input_ids'])}")

In [ ]:
# Cell 4 - Load model (4-bit). torch_dtype=bfloat16 BAT BUOC.
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=get_bnb_config(),
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False  # required for gradient_checkpointing

print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Cell 5 - Apply LoRA
lora_cfg = get_lora_config()
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# Cell 6 - Load val items for MAE callback
print(f"Loading {VAL_EVAL_SIZE} val items for MAE callback...")
val_items = load_items("validation", size=VAL_EVAL_SIZE)
print(f"  Loaded: {len(val_items)} items")
print(f"  Price range: {min(i.price for i in val_items):.0f}K - {max(i.price for i in val_items):.0f}K VND")

In [ ]:
# Cell 7 - Build callback + collator + trainer
mae_callback = MaeEvalCallback(
    model=model,
    tokenizer=tokenizer,
    val_items=val_items,
    output_dir=OUTPUT_DIR,
)
collator = DataCollatorCompletionOnly(tokenizer)
sft_cfg  = get_sft_config(OUTPUT_DIR, HUB_MODEL_ID)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    data_collator=collator,
    args=sft_cfg,
    callbacks=[mae_callback],
)
print("Trainer ready.")

In [ ]:
# Cell 8 - W&B init + train
wandb.init(project="qwen-vn-pricer-v2", name="v2-full-run")

trainer.train()

print("\n=== MAE History ===")
for step, mae in mae_callback.history:
    marker = " <-- BEST" if step == mae_callback.best_step else ""
    print(f"  step={step:>5}  mae={mae:.2f}K VND  ({mae*1000:,.0f} VND){marker}")
print(f"\nBest checkpoint at step {mae_callback.best_step}: MAE={mae_callback.best_mae:.2f}K VND")

In [ ]:
# Cell 9 - Reload best_mae_checkpoint + push (do NOT push trainer.model)
del model, trainer
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=get_bnb_config(),
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
best_model = PeftModel.from_pretrained(base, f"{OUTPUT_DIR}/best_mae_checkpoint")
best_model.push_to_hub(HUB_MODEL_ID, private=True)

print(f"Pushed BEST ckpt (step {mae_callback.best_step}, MAE {mae_callback.best_mae:.2f}K) to: https://huggingface.co/{HUB_MODEL_ID}")
wandb.finish()